### **정적 웹 수집을 이용한 공공데이터와 SNS 데이터(Rest API) 수집**

> **한눈에 보기**
> - 서울시 버스·서울 열린데이터광장(XML), 네이버 검색(JSON) 오픈 API 호출
> - 인증키는 환경변수로, 남의 글인 검색 결과는 출력 앞부분만
> - 정리 문서: [오픈API수집.md](../오픈API수집.md)

> 한화내일아카데미 10월 13일 수업 실습 노트북입니다. (강사님 자료 `dataexam/exam2.ipynb`)
> - **📝 정리** 셀은 실행 결과를 보고 직접 정리한 내용입니다.
> - 실습에서 만든 결과 파일(`output/` 아래 CSV, TXT, 이미지)은 다른 사이트의 글과 자료를 수집한 것이라 저장소에 올리지 않았습니다.
> - **API 키 처리**: 원본 노트북에는 공공데이터포털·서울 열린데이터광장·네이버 API 키가 코드와 출력(URL)에 그대로 들어 있었습니다. 이 사본에서는 키를 환경변수(`DATA_GO_KR_SERVICE_KEY`, `SEOUL_OPENAPI_KEY`, `NAVER_CLIENT_ID`, `NAVER_CLIENT_SECRET`)에서 읽도록 바꾸고, 출력에 찍힌 키는 `<...>` 자리표시자로 바꿨습니다.
> - 네이버 블로그/뉴스 검색 결과는 다른 사람이 쓴 글의 일부라 앞부분만 남겼습니다.

### 설치후 kernel restart 하세요...

In [1]:
%pip install lxml

  Using cached lxml-6.1.3-cp312-cp312-win_amd64.whl.metadata (3.4 kB)
Using cached lxml-6.1.3-cp312-cp312-win_amd64.whl (4.0 MB)
Note: you may need to restart the kernel to use updated packages.


In [1]:
from bs4 import BeautifulSoup
import urllib.request as req

busNum = '360'
import os
# 원래는 공공데이터포털 인증키(serviceKey)를 문자열로 직접 써 두었다. 깃허브에 올리면 키가 노출되므로 환경변수에서 읽도록 바꿨다.
key = os.environ.get("DATA_GO_KR_SERVICE_KEY")
url1 = 'http://ws.bus.go.kr/api/rest/busRouteInfo/getBusRouteList?serviceKey='+key+'&strSrch='+busNum
res = req.urlopen(url1)

soup = BeautifulSoup(res.read().decode('utf-8'), "xml")

busRouteId = None
for itemList in soup.find_all('itemList') :
    busRouteId = itemList.find('busRouteId').string
    busRouteNm = itemList.find('busRouteNm').string
    if busRouteNm == busNum :
        break

url2 = 'http://ws.bus.go.kr/api/rest/busRouteInfo/getStaionByRoute?ServiceKey='+key+'&busRouteId='+busRouteId
res = req.urlopen(url2)
soup = BeautifulSoup(res.read(), "xml")

busPos = []
for itemList in soup.find_all('itemList') :
    gpsY = itemList.find('gpsY').string
    gpsX = itemList.find('gpsX').string

    busPos.append((gpsY, gpsX))

print('[ ' + busNum + '번 버스의 운행 위치 ]')
if len(busPos) != 0 :
    print(busNum + '번 버스 ' + str(len(busPos)) + '대 운행중...')
    for lat,lng in busPos :
        print(lat+','+lng)
else :
    print('현재 운행중인 ' + busNum + '번 버스가 없어요...')

# URL에 인증키가 들어 있어서, 출력할 때는 키 부분을 가린다.
print(url1.replace(key, "<DATA_GO_KR_SERVICE_KEY>"))
print(url2.replace(key, "<DATA_GO_KR_SERVICE_KEY>"))

[ 360번 버스의 운행 위치 ]
360번 버스 91대 운행중...
37.4699297672,127.1275897549
37.478856,127.126126
37.481955,127.124583
37.4882127127,127.1214198552
37.494271,127.117262
37.500093,127.111934
37.503242,127.109062
37.5068001738,127.1058262006
37.512951,127.097765
37.5122132454,127.0939144028
37.5117722886,127.0872805398
37.5118706155,127.0833904794
37.51206,127.079682
37.5111175337,127.0720732937
37.5098983164,127.0659088469
37.508448,127.060899
37.5066831874,127.0554269279
37.505143,127.050205
37.503232,127.044057
37.50164,127.038772
37.500428,127.034735
37.49881,127.029558
37.500785,127.02637
37.506367,127.023728
37.51066,127.019358
37.5083481993,127.0120079425
37.506295,127.005137
37.5034578584,126.9956954487
37.5020371598,126.9899765867
37.503188597,126.9766525988
37.505867,126.971748
37.5064370015,126.9685844969
37.5075850724,126.9655921789
37.5115728985,126.9612375315
37.512734203,126.9509621786
37.5138325876,126.9447114504
37.5134800797,126.9388655754
37.5132368488,126.9323737344
37.513045,1

**📝 정리** — 서울시 버스 노선 API(XML)를 두 번 호출했다. ① `getBusRouteList`로 노선 번호 360의 노선 ID(`100100057`)를 찾고, ② 그 ID로 `getStaionByRoute`를 호출했다(`Staion`은 API 이름 자체의 철자). XML은 `BeautifulSoup(..., "xml")`로 파싱하는데, 이 파서를 쓰려면 앞 셀에서 설치한 `lxml`이 필요하다.

⚠️ 출력은 "360번 버스 91대 운행중"이지만, `getStaionByRoute`는 **노선이 지나는 정류소 목록**을 돌려주는 API라서 91은 버스 대수가 아니라 **정류소 개수**이고 좌표도 정류소 위치다. 실제 운행 중인 버스 위치를 보려면 버스 위치 정보 API를 따로 호출해야 한다. 코드의 문구가 결과와 맞지 않는 부분이다.

In [2]:
from bs4 import BeautifulSoup
import urllib.request as req

import os
# 원래는 서울 열린데이터광장 인증키를 문자열로 직접 써 두었다. 깃허브에 올리면 키가 노출되므로 환경변수에서 읽도록 바꿨다.
key = os.environ.get("SEOUL_OPENAPI_KEY")
contentType = 'xml'
startIndex = '1'
endIndex = '1000'
url = 'http://openapi.seoul.go.kr:8088/'+key+'/'+contentType+'/LampScpgmtb/'+startIndex+'/'+endIndex+'/'
# URL 경로에 인증키가 들어 있어서, 출력할 때는 키 부분을 가린다.
print(url.replace(key, "<SEOUL_OPENAPI_KEY>"))
res = req.urlopen(url)
xml = res.read()
soup = BeautifulSoup(xml, "xml")

pjList = []
for itemList in soup.find_all('row') :
    up_nm = itemList.find('UP_NM').string
    up_nm = '없음' if up_nm is None else up_nm
    pgm_nm = itemList.find('PGM_NM').string
    pgm_nm = '없음' if pgm_nm is None else pgm_nm
    target_nm = itemList.find('TARGET_NM').string
    target_nm = '없음' if target_nm is None else target_nm
    u_price = itemList.find('U_PRICE').string
    u_price = '없음' if u_price is None else u_price
    pjList.append((up_nm, pgm_nm, target_nm, u_price))

print('[ 서울 청소년 수련관 강좌 리스트 ]')
for up_nm,pgm_nm,target_nm,u_price in pjList :
    print(up_nm+','+pgm_nm+','+target_nm+','+str(u_price))

http://openapi.seoul.go.kr:8088/<SEOUL_OPENAPI_KEY>/xml/LampScpgmtb/1/1000/
[ 서울 청소년 수련관 강좌 리스트 ]
금천청소년수련관,아침B-청소년,주2회수영-청소년(구),36000
금천청소년수련관,난초A,초4~24세미만,41000
금천청소년수련관,아침C(5회)-청소년,주5회수영-청소년(구),56000
시립서대문청소년센터,경기민요와 장구 A,7세~초등,40000
시립서대문청소년센터,국가공인 한자지도사,성인,30000
시립서대문청소년센터,기린A,초2~5,30000
시립서대문청소년센터,상어1-3(초급),초등학생,40000
시립서대문청소년센터,셀프리더쉽&표현력 스피치 A,초3~6,33000
시립서대문청소년센터,속성 제빵자격증반,초6~성인(3개월),150000
시립서대문청소년센터,신나는미술세상(C),5~6세,30000
시립서대문청소년센터,신나는미술세상(D),초등학생,30000
시립서대문청소년센터,죠스2-1(상급),초등학생,33000
시립서대문청소년센터,죠스2-1(상급3개월) ,초등학생,89100
시립서대문청소년센터,죠스2-2(중급),초등학생,33000
광진청소년센터,목요바이올린(7세-성인)B,성인,61000
마포청소년수련관,재미있는통기타교실(토12시40분),초3-성인,35000
동대문청소년수련관,청소년성장집단상담,구분없음,1000000
시립서대문청소년센터,방학)체스(A),7세~초등,35000
시립서대문청소년센터,방학)체스(B),7세~초등,35000
시립서대문청소년센터,실버 (3개월),성인,청소년(14세부터),142560
금천청소년수련관,난초A중급-청소년,주3회수영-청소년,41000
금천청소년수련관,매화C(5회),초4~24세미만,56000
금천청소년수련관,물개유아A,만5세~초3,36000
창동청소년센터,9/7 1일특강 취미난타(성인),성인,10000
창동청소년센터,특)우드아트,청소년,10000
노원청소년센터,21학년-(수업료)-7세 사자,체르니30(청소년),610000
화곡청소년센터,미라 소세지 빵10/16(토)

**📝 정리** — 서울 열린데이터광장 API는 인증키를 쿼리 파라미터가 아니라 **URL 경로**(`/키/xml/서비스명/시작/끝/`)에 넣는다. 한 번에 최대 1000건(1~1000)을 요청했다. 값이 비어 있는 태그는 `.string`이 `None`이라 `'없음'`으로 바꿨다. 출력은 쉼표로 이어 붙였는데 "성인,청소년(14세부터)"처럼 값 안에 쉼표가 있는 경우가 있어서, 이대로 CSV로 저장하면 열이 어긋난다.

In [3]:
from bs4 import BeautifulSoup
import urllib.request as req
import io

import os
# 원래는 서울 열린데이터광장 인증키를 문자열로 직접 써 두었다. 깃허브에 올리면 키가 노출되므로 환경변수에서 읽도록 바꿨다.
key = os.environ.get("SEOUL_OPENAPI_KEY")
contentType = 'xml'
startIndex = '1'
endIndex = '1000'
date = '20261009'

url = 'http://openapi.seoul.go.kr:8088/'+key+'/'+contentType+'/CardSubwayStatsNew/'+startIndex+'/'+endIndex+'/'+date+'/'
# URL 경로에 인증키가 들어 있어서, 출력할 때는 키 부분을 가린다.
print(url.replace(key, "<SEOUL_OPENAPI_KEY>"))
res = req.urlopen(url)
xml = res.read()

soup = BeautifulSoup(xml, "xml")

subwayList = []
for itemList in soup.find_all('row') :
    line_num = itemList.find('SBWY_ROUT_LN_NM').string
    sub_sta_nm = itemList.find('SBWY_STNS_NM').string
    ride_pasgr_num = itemList.find('GTON_TNOPE').string
    alight_pasgr_num = itemList.find('GTOFF_TNOPE').string
    subwayList.append((line_num, sub_sta_nm, ride_pasgr_num, alight_pasgr_num))

print(len(subwayList))
print('[ 서울시 지하철호선별 역별 승하차 인원 정보 ]')
for line_num, sub_sta_nm, ride_pasgr_num, alight_pasgr_num in subwayList :
    print(line_num+','+sub_sta_nm+','+ride_pasgr_num+','+alight_pasgr_num)


http://openapi.seoul.go.kr:8088/<SEOUL_OPENAPI_KEY>/xml/CardSubwayStatsNew/1/1000/20261009/
616
[ 서울시 지하철호선별 역별 승하차 인원 정보 ]
1호선,서울역,80561,77550
1호선,시청,23558,25989
1호선,종각,31143,28769
1호선,종로3가,29436,27141
1호선,종로5가,23607,23277
1호선,동대문,15993,16044
1호선,신설동,10981,10075
1호선,제기동,15683,16052
1호선,청량리(서울시립대입구),22839,23543
1호선,동묘앞,13701,15131
2호선,시청,17255,15190
2호선,을지로입구,40279,38986
2호선,을지로3가,18513,18721
2호선,을지로4가,11335,11137
2호선,동대문역사문화공원(DDP),17003,19428
2호선,신당,12680,13295
2호선,상왕십리,10238,9803
2호선,왕십리(성동구청),13012,11789
2호선,한양대,5833,5991
2호선,뚝섬,19319,19653
2호선,성수,58602,65384
2호선,건대입구,41232,40091
2호선,구의(광진구청),21399,20326
2호선,강변(동서울터미널),26669,26562
2호선,잠실나루,9098,8983
2호선,잠실(송파구청),80960,77605
2호선,잠실새내,18327,18249
2호선,종합운동장,7131,7898
2호선,삼성(무역센터),28908,27594
2호선,선릉,24835,22119
2호선,역삼,16181,18012
2호선,강남,48529,47934
2호선,교대(법원.검찰청),13919,17720
2호선,서초,9780,8142
2호선,방배,8932,9112
2호선,사당,35142,39306
2호선,낙성대(강감찬),22590,20161
2호선,서울대입구(관악구청),36107,33592
2호선,봉천,17803,16004
2호선,신림,45272,43311
2호선,신대방,20336,18284

**📝 정리** — 같은 방식으로 2026년 10월 9일 지하철 호선별·역별 승하차 인원을 조회해 616건을 받았다. 날짜는 URL 맨 끝 경로에 넣는다. (홍대입구 2호선 하차 96,136명, 서울역 1호선 승차 80,561명 등)

In [4]:
import requests
import urllib
import json
import os
# 원래는 네이버 개발자센터 Client ID/Secret을 문자열로 직접 써 두었다. 깃허브에 올리면 키가 노출되므로 환경변수에서 읽도록 바꿨다.
client_key = os.environ.get("NAVER_CLIENT_ID")
client_secret = os.environ.get("NAVER_CLIENT_SECRET")
query = '스마트폰'
encText = urllib.parse.quote_plus(query)
num = 100
naver_url = 'https://openapi.naver.com/v1/search/blog.json?query=' + encText + '&display=' + str(num)

naver_auth_header = {"X-Naver-Client-Id":client_key, "X-Naver-Client-Secret":client_secret}
response = requests.get(naver_url, headers=naver_auth_header)
rescode = response.status_code
if(rescode == 200):
    data_dict = response.json()

    print('[' + query + '에 대한 네이버 블로그 글 ]')
    for count, data in enumerate(data_dict['items'], 1) :
        print (str(count) + ' : ' + data['title'])
        print ('[' + data['description'] + ']')
else:
    print('오류 코드 : ' + rescode)

[스마트폰에 대한 네이버 블로그 글 ]
1 : 내 <b>스마트폰</b> 최고 성능으로 만드는 설정 비법
[알림 설정 <b>스마트</b>하게 관리하기 6. 보안 설정 강화로 개인 정보 보호 7. 데이터 사용량 절약 꿀팁 앱과 시스템 속도 향상의 비밀 <b>스마트폰</b>이 느려졌다고 느껴질 때, 가장 먼저 떠올리는 것은 아마도 용량... ]
2 : <b>스마트폰</b> hwp 파일 열기 한글파일
[<b>스마트폰</b> hwp 파일 열기 방법 에 대해서 알아보도록 하겠습니다. hwp... 간단하게 <b>스마트폰</b> hwp파일 열기 방법에 대해서 알아보는데 아이폰... 아이폰 <b>스마트폰</b> hwp 파일 열기 안드로이드 <b>스마트폰</b> hwp 파일 열기... ]
3 : [알라딘서재]<b>스마트폰</b>이 된 멜리스
[커다란 눈으로 <b>스마트폰</b>을 들고 독자를 응시하는 소녀와 붉은 글씨로 &quot;<b>스마트폰</b>&quot;제목이 보이며 <b>스마트폰</b>이 중요하고 위험하다는 경고성을 알려주는 느낌을 받았다. 과연 멜리스는 어떻게 <b>스마트폰</b>이 되었을까?... ]
... (이하 생략 — 블로그 글 100건 중 앞 3건만 남김. 다른 사람이 쓴 글의 일부라 저장소에는 줄여서 올림)


**📝 정리** — 네이버 검색 API는 인증 정보를 URL이 아니라 **요청 헤더**(`X-Naver-Client-Id`, `X-Naver-Client-Secret`)로 보낸다. `display=100`으로 블로그 글 100건을 받았다. 제목과 요약에 검색어 강조용 `<b>` 태그가 섞여 있다.

⚠️ `else` 쪽 `'오류 코드 : ' + rescode`는 `rescode`가 정수라서 실제로 오류가 나면 `TypeError`가 난다. `str(rescode)`로 바꿔야 한다.

In [5]:
import urllib.request
import json
import os
# 원래는 네이버 개발자센터 Client ID/Secret을 문자열로 직접 써 두었다. 깃허브에 올리면 키가 노출되므로 환경변수에서 읽도록 바꿨다.
client_key = os.environ.get("NAVER_CLIENT_ID")
client_secret = os.environ.get("NAVER_CLIENT_SECRET")
query = '스마트폰'
encText = urllib.parse.quote_plus(query)
num = 100
naver_url = 'https://openapi.naver.com/v1/search/blog.json?query=' + encText + '&display=' + str(num)

request = urllib.request.Request(naver_url)
request.add_header("X-Naver-Client-Id",client_key)
request.add_header("X-Naver-Client-Secret",client_secret)
response = urllib.request.urlopen(request)
rescode = response.getcode()
if(rescode == 200):
    response_body = response.read()
    data_dict = json.loads(response_body)

    print('[' + query + '에 대한 네이버 블로그 글 ]')
    for count, data in enumerate(data_dict['items'], 1) :
        print (str(count) + ' : ' + data['title'])
        print ('[' + data['description'] + ']')
else:
    print('오류 코드 : ' + rescode)

[스마트폰에 대한 네이버 블로그 글 ]
1 : 내 <b>스마트폰</b> 최고 성능으로 만드는 설정 비법
[알림 설정 <b>스마트</b>하게 관리하기 6. 보안 설정 강화로 개인 정보 보호 7. 데이터 사용량 절약 꿀팁 앱과 시스템 속도 향상의 비밀 <b>스마트폰</b>이 느려졌다고 느껴질 때, 가장 먼저 떠올리는 것은 아마도 용량... ]
2 : <b>스마트폰</b> 카메라 셔터 소리 끄기 친절한 안내
[안드로이드 <b>스마트폰</b> 셔터 소리 끄기 3. 아이폰 <b>스마트폰</b> 셔터 소리 끄기 4. 국가별 규제로 인한 셔터... 우리가 <b>스마트폰</b>으로 사진을 찍을 때, '찰칵' 하는 경쾌한 소리는 마치 사진이 잘 찍혔다는 신호처럼... ]
3 : <b>스마트폰</b> 아이콘을 좀 더 개성있게 옵티머스 LTE3 내맘대로 아이콘 3.....
[<b>스마트폰</b> 아이콘을 좀 더 개성있게 옵티머스 LTE3 내맘대로 아이콘 3.0 2013.03.19 07:00 Tech/<b>스마트폰</b> http://www.fillresearch.com 광고 관절 통증엔... 다른 <b>스마트폰</b>과 다르게 전용 단말기만의 조금 재미있는 기능을... ]
... (이하 생략 — 블로그 글 100건 중 앞 3건만 남김. 다른 사람이 쓴 글의 일부라 저장소에는 줄여서 올림)


**📝 정리** — 같은 요청을 `urllib.request.Request` + `add_header()`로 보냈다. (`urllib`은 200이 아닌 응답이면 `else`까지 가기 전에 `HTTPError` 예외를 내므로, 오류 처리는 `try/except`로 하는 게 맞다.) 같은 검색인데 위 셀과 2번째 글부터 순서가 달랐다.

In [6]:
data_dict # JSON

{'lastBuildDate': 'Tue, 13 Oct 2026 15:14:20 +0900',
 'total': 23004542,
 'start': 1,
 'display': 100,
 'items': [{'title': '내 <b>스마트폰</b> 최고 성능으로 만드는 설정 비법',
   'link': 'https://fuelurwanderlust.tistory.com/5',
   'description': '알림 설정 <b>스마트</b>하게 관리하기 6. 보안 설정 강화로 개인 정보 보호 7. 데이터 사용량 절약 꿀팁 앱과 시스템 속도 향상의 비밀 <b>스마트폰</b>이 느려졌다고 느껴질 때, 가장 먼저 떠올리는 것은 아마도 용량... ',
   'bloggername': 'Fuel your Wanderlust',
   'bloggerlink': 'https://fuelurwanderlust.tistory.com/',
   'postdate': '20251106'},
 ... (100건 중 첫 번째 글만 남기고 생략)

**📝 정리** — 응답 JSON에는 `total`(전체 검색 결과 수 23,004,542건), `display`, `items`가 있고, 글마다 `title`, `link`, `description`, `bloggername`, `postdate`가 들어 있다.

In [7]:
data_dict["items"][0]["description"]

'알림 설정 <b>스마트</b>하게 관리하기 6. 보안 설정 강화로 개인 정보 보호 7. 데이터 사용량 절약 꿀팁 앱과 시스템 속도 향상의 비밀 <b>스마트폰</b>이 느려졌다고 느껴질 때, 가장 먼저 떠올리는 것은 아마도 용량... '

In [8]:
title = []
desc = []
for data in data_dict["items"]:
    title.append(data["title"])
    desc.append(data["description"])  

In [9]:
len(title)

100

In [10]:
len(desc)

100

In [11]:
title

['내 <b>스마트폰</b> 최고 성능으로 만드는 설정 비법',
 '<b>스마트폰</b> 카메라 셔터 소리 끄기 친절한 안내',
 '<b>스마트폰</b> 아이콘을 좀 더 개성있게 옵티머스 LTE3 내맘대로 아이콘 3.....',
 '<b>스마트폰</b> 화면이 제멋대로 돌아갈 때 알아두면 좋은 몇 가지',
 '제천여행 <b>스마트폰</b> 앱 활용 스탬프투어 문화관광 마일리지 하나로 ....',
 ... (100건 중 앞 5건만 남기고 생략)

In [12]:
import re
for i in range(len(title)) :
   title[i] = re.sub("</?b>", "", title[i])

In [13]:
for i in range(len(desc)) :
   desc[i] = re.sub("</?b>", "", desc[i])

**📝 정리** — 정규식 `</?b>`로 `<b>`, `</b>` 태그를 지웠다. 하지만 `&amp;`, `&quot;` 같은 HTML 엔티티는 그대로 남는다(예: `사진&amp;영상`). `html.unescape()`를 같이 쓰면 원래 글자로 바뀐다.

In [14]:
import pandas as pd
dictdata = {"title":title,"description":desc}
df = pd.DataFrame(dictdata)
df

,title,description
0,내 스마트폰 최고 성능으로 만드는 설정 비법,알림 설정 스마트하게 관리하기 6. 보안 설정 강화로 개인 정보 보호 7. 데이터 ...
1,스마트폰 카메라 셔터 소리 끄기 친절한 안내,안드로이드 스마트폰 셔터 소리 끄기 3. 아이폰 스마트폰 셔터 소리 끄기 4. 국가...
2,스마트폰 아이콘을 좀 더 개성있게 옵티머스 LTE3 내맘대로 아이콘 3.....,스마트폰 아이콘을 좀 더 개성있게 옵티머스 LTE3 내맘대로 아이콘 3.0 2013...
3,스마트폰 화면이 제멋대로 돌아갈 때 알아두면 좋은 몇 가지,스마트폰을 쓰다 보면 화면이 제멋대로 돌아가 당황스러울 때가... 스마트폰 자동 회...
4,제천여행 스마트폰 앱 활용 스탬프투어 문화관광 마일리지 하나로 ....,국내여행/충청북도 제천여행 스마트폰 앱 활용 스탬프투어 문화관광 마일리지 하나로 여...
...,...,...
95,아이폰 lg 스마트tv 미러링 엘지티비 핸드폰연결 방법,"안드로이드 스마트폰인 갤럭시 같은 경우에는 연동하는 방법이 쉽고 간단해, 누구나 쉽..."
96,삼성 갤럭시 스마트폰 배터리 수명 교체 주기 비용 확인,요즘 스마트폰을 한 번 구매하면 최소 2년 정도 사용할 수 있습니다.... 그래서 ...
97,"스마트폰 핸드폰 타자연습, 휴대폰 자판 키보드 연습앱 어플 추천","급감, 스마트 키보드 기본 키보드와 달라 적응 필요 스마트폰 타자를 처음 배우는 느..."
98,"아이폰 SE3, 갤럭시 S24 장단점 비교, 가성비폰은 어떤게 효과적일까?",스마트폰을 구매할 때 가장 많이 고민하는 부분 중 하나가 바로... 과연 두 제품 ...


In [15]:
df.to_csv("output/smartphone.csv")

**📝 정리** — `to_csv()`는 기본으로 인덱스 열까지 저장해서 다시 읽으면 `Unnamed: 0` 열이 생긴다(`index=False`로 뺄 수 있다). 엑셀에서 한글이 깨지지 않게 하려면 `encoding='utf-8-sig'`를 준다.

In [16]:
import urllib.request
import json

import os
# 원래는 네이버 개발자센터 Client ID/Secret을 문자열로 직접 써 두었다. 깃허브에 올리면 키가 노출되므로 환경변수에서 읽도록 바꿨다.
client_key = os.environ.get("NAVER_CLIENT_ID")
client_secret = os.environ.get("NAVER_CLIENT_SECRET")
query = '아시안게임'
encText = urllib.parse.quote_plus(query)

num = 100
naver_url = 'https://openapi.naver.com/v1/search/news.json?query=' + encText + '&display=' + str(num)

request = urllib.request.Request(naver_url)
request.add_header("X-Naver-Client-Id",client_key)
request.add_header("X-Naver-Client-Secret",client_secret)
response = urllib.request.urlopen(request)

rescode = response.getcode()

if(rescode == 200):
    response_body = response.read()
    dataList = json.loads(response_body)
    count = 1
    print('[' + query + '에 대한 네이버 뉴스 글 ]')
    for data in dataList['items'] :
        print (str(count) + ' : ' + data['title'])
        print ('[' + data['description'] + ']')
        count += 1
else:
    print('오류 코드 : ' + rescode)

[아시안게임에 대한 네이버 뉴스 글 ]
1 : '한화 왕서방', 내년에도 쭈욱 '한화맨'으로
[정규시즌 26경기 133.2이닝, 11승 6패 100삼진, ERA 3.97로 지난 2026 아이치·나고야 <b>아시안게임</b>에선 대만 대표로 출전까지 했다. 한화 입단 당시만 해도 이 정도까지 성적을 내주리란 예상은 많지 않았다. AQ로 들어올 수 있는... ]
2 : 전국체전 맞아 제주서 한국체육학회 학술대회…‘국민 스포츠권’ 논의
[16일 심포지엄에서는 이상현 2026 아이치·나고야 <b>아시안게임</b> 대한민국 선수단 단장이 대회 성과와 과제를 발표한다. 송명근 대한체육회 공정법무실장은 전문체육과 생활체육 통합 10년을 돌아보고, 박재현... ]
3 : [오피셜] 류지현 감독 '1호 선물'은 왕옌청 재계약!…연봉 2배 상승→총...
[2026 아이치·나고야 <b>아시안게임</b>에 대만 대표팀으로 뛴 그는 입국 후 시즌을 조기에 마감했다. 올 시즌 왕옌청은 26경기에 출전해 133⅔이닝을 책임지며 11승 6패 평균자책점 3.97, 100탈삼진 등의 기록을 남겼다. 한화는... ]
... (이하 생략 — 뉴스 기사 100건 중 앞 3건만 남김. 다른 사람이 쓴 글의 일부라 저장소에는 줄여서 올림)


**📝 정리** — 뉴스 검색은 URL만 `blog.json` → `news.json`으로 바꾸면 된다. 같은 키로 여러 검색 API를 쓸 수 있다.

## <span style='color:blue; font-weight:bold'>오픈 API를 이용한 데이터 수집에 대하여 경험해 보았습니다.</span>

## 🔁 복습 질문

<details>
<summary>1. XML 응답에서 항목 목록을 꺼내는 코드는?</summary>

`BeautifulSoup(text, "xml").find_all("itemList")`(버스) 또는 `find_all("row")`(열린데이터광장).
</details>

<details>
<summary>2. JSON 응답을 파이썬 dict로 바꾸는 방법은?</summary>

`response.json()`.
</details>
